**DO NOT RUN FOR SUPERVISED TRAINING:** The target parser does not read the mounted xBD WKT annotations. Previous checkpoints are invalid for supervised evaluation. Fix and remotely validate the target parser first.

# V4 DINOv3 ViT-L D0 remote smoke

This notebook is Kaggle-only. It loads the already-profiled satellite ViT-L checkpoint as a frozen backbone, creates a deterministic event-grouped xBD train/validation split, trains a small paired temporal dense head, validates finite losses, and writes a checkpoint/report. `RV4_MAX_SAMPLES` and `RV4_EPOCHS` keep this gate bounded; full training is a separate authorized run after this smoke passes.


In [ ]:
from __future__ import annotations

raise RuntimeError("Blocked: this historical notebook does not parse xBD WKT targets; do not train or reuse its checkpoints")

import hashlib
import importlib.util
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from torch import nn
from torch.nn import functional as F

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/rv4_dinov3_d0_smoke.json")
CHECKPOINT_PATH = Path("/kaggle/working/rv4_dinov3_d0_smoke.pt")
MODEL_NAME = "dinov3_vitl16"
MODEL_REVISION = "facebook/dinov3-vitl16-pretrain-sat493m"
PATCH_SIZE = 16
CROP_SIZE = 512
SEED = 42
MAX_SAMPLES = int(os.environ.get("RV4_MAX_SAMPLES", "8"))
EPOCHS = int(os.environ.get("RV4_EPOCHS", "1"))


def find_model_inputs() -> tuple[Path, Path]:
    hubconf_paths = sorted(INPUT_ROOT.rglob("hubconf.py"))
    if not hubconf_paths:
        raise RuntimeError("the attached DINOv3 repository has no hubconf.py")
    weights = sorted(INPUT_ROOT.rglob("*.pth"))
    if not weights:
        raise RuntimeError("the attached DINOv3 ViT-L checkpoint is missing")
    return hubconf_paths[0].parent, weights[0]


def find_xbd_pairs() -> list[dict[str, object]]:
    records: list[dict[str, object]] = []
    for pre_path in sorted(INPUT_ROOT.rglob("*_pre_disaster.png")):
        post_path = pre_path.with_name(pre_path.name.replace("_pre_disaster.png", "_post_disaster.png"))
        if not post_path.exists():
            continue
        split_dir = pre_path.parent.parent
        split = split_dir.name
        if split not in {"train", "test"}:
            continue
        sample_id = pre_path.name[: -len("_pre_disaster.png")]
        label_path = split_dir / "labels" / f"{sample_id}_post_disaster.json"
        if not label_path.exists() or split != "train":
            continue
        event_id, separator, tail = sample_id.rpartition("_")
        if not separator or not tail.isdigit():
            event_id = sample_id
        records.append({
            "sample_id": sample_id,
            "event_id": event_id,
            "pre": pre_path,
            "post": post_path,
            "label": label_path,
        })
    if not records:
        raise RuntimeError("no labeled xBD train pairs were found")
    return records


def read_target(label_path: Path, width: int, height: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    payload = json.loads(label_path.read_text(encoding="utf-8"))
    loc = Image.new("L", (width, height), 0)
    damage = Image.new("L", (width, height), 255)
    draw_loc = ImageDraw.Draw(loc)
    draw_damage = ImageDraw.Draw(damage)
    classes = {"no-damage": 0, "minor-damage": 1, "major-damage": 2, "destroyed": 3}
    raw_features = payload.get("features", [])
    if isinstance(raw_features, dict):
        raw_features = next(
            (raw_features[key] for key in ("xy", "lng_lat", "features") if isinstance(raw_features.get(key), list)),
            [],
        )
    if not isinstance(raw_features, list):
        raise RuntimeError(f"xBD features must be a list or mapping of lists: {label_path}")
    for feature in raw_features:
        if not isinstance(feature, dict):
            continue
        properties = feature.get("properties") or {}
        if properties.get("feature_type", "building") not in {"building", "building_polygon", "building-footprint"}:
            continue
        geometry = feature.get("geometry") or {}
        polygons = geometry.get("coordinates", [])
        if geometry.get("type") == "Polygon":
            polygons = [polygons]
        if geometry.get("type") != "MultiPolygon" and geometry.get("type") != "Polygon":
            continue
        subtype = properties.get("subtype")
        if isinstance(subtype, str):
            subtype = subtype.strip().lower().replace("_", "-").replace(" ", "-")
        class_id = classes.get(subtype)
        for polygon in polygons:
            if not polygon:
                continue
            points = [(float(point[0]), float(point[1])) for point in polygon[0] if len(point) >= 2]
            if len(points) < 3:
                continue
            draw_loc.polygon(points, fill=1)
            if class_id is not None:
                draw_damage.polygon(points, fill=class_id)
    loc_array = np.asarray(loc, dtype=np.float32)
    damage_array = np.asarray(damage, dtype=np.int64)
    damage_valid = damage_array != 255
    return loc_array, damage_array, damage_valid


def load_crop(path: Path, top: int, left: int) -> torch.Tensor:
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    crop = array[top : top + CROP_SIZE, left : left + CROP_SIZE]
    if crop.shape != (CROP_SIZE, CROP_SIZE, 3):
        raise RuntimeError(f"crop shape mismatch for {path}: {crop.shape}")
    tensor = torch.from_numpy(crop).permute(2, 0, 1)
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=tensor.dtype).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=tensor.dtype).view(3, 1, 1)
    return (tensor - mean) / std


def make_sample(record: dict[str, object], *, training: bool, rng: random.Random) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    pre_path = record["pre"]
    post_path = record["post"]
    label_path = record["label"]
    with Image.open(pre_path) as image:
        width, height = image.size
    if training:
        top = rng.randint(0, max(0, height - CROP_SIZE))
        left = rng.randint(0, max(0, width - CROP_SIZE))
    else:
        top = max(0, (height - CROP_SIZE) // 2)
        left = max(0, (width - CROP_SIZE) // 2)
    loc, damage, damage_valid = read_target(label_path, width, height)
    loc = torch.from_numpy(loc[top : top + CROP_SIZE, left : left + CROP_SIZE])
    damage = torch.from_numpy(damage[top : top + CROP_SIZE, left : left + CROP_SIZE])
    damage_valid = torch.from_numpy(damage_valid[top : top + CROP_SIZE, left : left + CROP_SIZE])
    return load_crop(pre_path, top, left), load_crop(post_path, top, left), loc, damage, damage_valid


class TemporalHead(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2048, 256, kernel_size=1),
            nn.GroupNorm(32, 256),
            nn.GELU(),
            nn.Conv2d(256, 5, kernel_size=1),
        )

    def forward(self, pre: torch.Tensor, post: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([pre, post], dim=1))


def tokens_to_grid(tokens: torch.Tensor) -> torch.Tensor:
    expected = (CROP_SIZE // PATCH_SIZE) ** 2
    if tokens.ndim != 3 or tokens.shape[1] != expected:
        raise RuntimeError(f"expected [B,{expected},C] patch tokens, got {tuple(tokens.shape)}")
    side = CROP_SIZE // PATCH_SIZE
    return tokens.reshape(tokens.shape[0], side, side, tokens.shape[-1]).permute(0, 3, 1, 2).contiguous()


def loss_for_batch(logits: torch.Tensor, loc: torch.Tensor, damage: torch.Tensor, damage_valid: torch.Tensor) -> torch.Tensor:
    logits = F.interpolate(logits, size=(CROP_SIZE, CROP_SIZE), mode="bilinear", align_corners=False)
    loc_logits = logits[:, :1]
    damage_logits = logits[:, 1:]
    loc_target = loc.unsqueeze(1).to(dtype=loc_logits.dtype)
    loc_loss = F.binary_cross_entropy_with_logits(loc_logits, loc_target)
    safe_damage = damage.masked_fill(~damage_valid, 0).to(dtype=torch.long)
    if damage_valid.any():
        damage_loss = F.cross_entropy(damage_logits, safe_damage, reduction="none")[damage_valid].mean()
    else:
        damage_loss = damage_logits.sum() * 0.0
    return 0.5 * loc_loss + 0.5 * damage_loss


def load_backbone(repo_root: Path, weights: Path) -> nn.Module:
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    model = torch.hub.load(str(repo_root), MODEL_NAME, source="local", weights=str(weights))
    return model.eval()


def extract_tokens(model: nn.Module, image: torch.Tensor, device: torch.device) -> torch.Tensor:
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model.forward_features(image.to(device)) if hasattr(model, "forward_features") else model(image.to(device))
    if isinstance(output, dict):
        tokens = output.get("x_norm_patchtokens")
    else:
        tokens = getattr(output, "x_norm_patchtokens", None)
    if not torch.is_tensor(tokens):
        raise RuntimeError("DINOv3 output did not expose x_norm_patchtokens")
    return tokens.clone().float()


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("D0 smoke requires CUDA; refusing a CPU fallback")
    if EPOCHS <= 0 or MAX_SAMPLES == 0:
        raise RuntimeError("D0 smoke requires positive RV4_EPOCHS and RV4_MAX_SAMPLES")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device("cuda")
    repo_root, weights = find_model_inputs()
    records = find_xbd_pairs()
    events = sorted({str(record["event_id"]) for record in records})
    validation_events = {event for index, event in enumerate(events) if index % 5 == 0}
    train_records = [record for record in records if str(record["event_id"]) not in validation_events]
    validation_records = [record for record in records if str(record["event_id"]) in validation_events]
    if not train_records or not validation_records:
        raise RuntimeError("event-grouped train/validation split is empty")
    if MAX_SAMPLES > 0:
        train_records = train_records[:MAX_SAMPLES]
        validation_records = validation_records[: max(1, min(2, MAX_SAMPLES // 4 or 1))]
    backbone = load_backbone(repo_root, weights).to(device)
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    head = TemporalHead().to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-4, weight_decay=1e-4)
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    train_losses: list[float] = []
    optimizer_steps = 0
    for epoch in range(EPOCHS):
        head.train()
        for index, record in enumerate(train_records):
            pre, post, loc, damage, damage_valid = make_sample(record, training=True, rng=random.Random(SEED + epoch * 100000 + index))
            pre_tokens = extract_tokens(backbone, pre.unsqueeze(0), device)
            post_tokens = extract_tokens(backbone, post.unsqueeze(0), device)
            logits = head(tokens_to_grid(pre_tokens), tokens_to_grid(post_tokens))
            loss = loss_for_batch(logits, loc.unsqueeze(0).to(device), damage.unsqueeze(0).to(device), damage_valid.unsqueeze(0).to(device))
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite D0 smoke loss at sample {index}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            optimizer.step()
            optimizer_steps += 1
            train_losses.append(float(loss.detach().cpu()))
    head.eval()
    validation_losses: list[float] = []
    with torch.inference_mode():
        for index, record in enumerate(validation_records):
            pre, post, loc, damage, damage_valid = make_sample(record, training=False, rng=random.Random(SEED + index))
            pre_tokens = extract_tokens(backbone, pre.unsqueeze(0), device)
            post_tokens = extract_tokens(backbone, post.unsqueeze(0), device)
            logits = head(tokens_to_grid(pre_tokens), tokens_to_grid(post_tokens))
            loss = loss_for_batch(logits, loc.unsqueeze(0).to(device), damage.unsqueeze(0).to(device), damage_valid.unsqueeze(0).to(device))
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite D0 validation loss at sample {index}")
            validation_losses.append(float(loss.detach().cpu()))
    torch.save({"model_revision": MODEL_REVISION, "head": head.state_dict(), "seed": SEED}, CHECKPOINT_PATH)
    report = {
        "schema_version": "rv4-dinov3-d0-smoke-1",
        "status": "PASS",
        "model_revision": MODEL_REVISION,
        "weights": str(weights),
        "repository": str(repo_root),
        "gpu_name": torch.cuda.get_device_name(device),
        "torch_version": torch.__version__,
        "train_records_available": len(records) - len([record for record in records if str(record["event_id"]) in validation_events]),
        "validation_records_available": len(validation_records),
        "train_samples_run": len(train_records) * EPOCHS,
        "validation_samples_run": len(validation_records),
        "epochs": EPOCHS,
        "optimizer_steps": optimizer_steps,
        "train_loss_last": train_losses[-1],
        "train_loss_mean": float(np.mean(train_losses)),
        "validation_loss_mean": float(np.mean(validation_losses)),
        "checkpoint": str(CHECKPOINT_PATH),
        "checkpoint_sha256": hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest(),
        "elapsed_seconds": time.perf_counter() - started,
        "peak_memory_allocated": int(torch.cuda.max_memory_allocated(device)),
        "peak_memory_reserved": int(torch.cuda.max_memory_reserved(device)),
    }
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()
